In [ ]:
# 1. Trong số các khách hàng có nhiều hơn một đơn hàng,
#  trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)
import pandas as pd
order = pd.read_csv("dataset/orders.csv", parse_dates=["order_date"]) # 
over1_orders_sorted = order.groupby("customer_id").filter(lambda x: len(x) > 1).sort_values(["customer_id", "order_date"])
over1_orders_sorted["gap"] = over1_orders_sorted.groupby("customer_id")["order_date"].diff().dt.days # .dt.days: lấy số ngày dạng nguyên từ "4 days 00:00:00"
gaps = over1_orders_sorted["gap"].dropna() # Bỏ NaN (dòng đầu mỗi customer)
print(f"Trung vị số ngày giữa 2 lần mua liên tiếp của khách mua nhiều đơn: {gaps.median()} ngày") 

Trung vị số ngày giữa 2 lần mua liên tiếp của khách mua nhiều đơn: 144.0 ngày


In [29]:
# 2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?
import pandas as pd
product = pd.read_csv("dataset/products.csv")
product["tsln_gop_tb"] = (product["price"] - product["cogs"])/product["price"]
tsln_goptb_theo_pk = product.groupby("segment")["tsln_gop_tb"].mean().sort_values(ascending=False)
print(tsln_goptb_theo_pk)
print(f"TSLN gộp trung bình cao nhất là: {tsln_goptb_theo_pk.idxmax()} với {tsln_goptb_theo_pk.max():.2f}")

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: tsln_gop_tb, dtype: float64
TSLN gộp trung bình cao nhất là: Standard với 0.31


In [11]:
# 3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), 
# lý do trả hàng nào xuất hiện nhiều nhất?
import pandas as pd
re = pd.read_csv("dataset/returns.csv"); pro = pd.read_csv("dataset/products.csv")
df = pd.merge(re, pro,on = "product_id", how = "right")
df = df[df["category"] == "Streetwear"]
df["return_reason"].value_counts()

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

In [21]:
# 4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ chuyển đổi trung bình (conversion_rate) cao nhất
#  trên tất cả các ngày mà nguồn đó xuất hiện?
import pandas as pd
web =  pd.read_csv("dataset/web_traffic.csv")
web = web.groupby("traffic_source")["bounce_rate"].mean().sort_values(ascending=False)
print(web)
print(f"Nguồn có tỉ lệ thoát trung bình thấp nhất: {web.idxmin()}")

traffic_source
direct            0.004511
organic_search    0.004504
referral          0.004499
paid_search       0.004478
social_media      0.004476
email_campaign    0.004458
Name: bounce_rate, dtype: float64
Nguồn có tỉ lệ thoát trung bình thấp nhất: email_campaign


In [ ]:
# 5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?
import pandas as pd
ord =  pd.read_csv("dataset/order_items.csv",dtype={"promo_id_2": "string"})
r = ord["promo_id"].notna().sum() / len(ord) * 100
print(f"{r:.2f}")


38.66


In [44]:
# 6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? 
# (tổng số đơn / số khách hàng trong nhóm)
import pandas as pd
cust = pd.read_csv("dataset/customers.csv"); orders = pd.read_csv("dataset/orders.csv")
cust_no_na_age = cust[cust["age_group"].notna()][["customer_id", "age_group"]]
orders = orders[["order_id","order_date", "customer_id"]]
df = pd.merge(cust_no_na_age, orders, on = "customer_id", how = "left")
num_order_by_age_group = df.groupby('age_group')["order_id"].count() # tổng số đơn theo từng nhóm tuổi
num_customer_by_age_group = df.groupby('age_group')["customer_id"].count()  # số khách theo từng nhóm tuổi - đếm 1 lần cho 1 khách
r = (num_order_by_age_group / num_customer_by_age_group).sort_values(ascending=False)
r


age_group
55+      0.954768
45-54    0.954049
35-44    0.953663
18-24    0.952512
25-34    0.952339
dtype: float64

In [ ]:
# 7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?
import pandas as pd
geography = pd.read_csv("dataset/geography.csv")
orders = pd.read_csv("dataset/orders.csv")
order_items = pd.read_csv("dataset/order_items.csv", dtype={"promo_id_2": "string"})
orders_geo = orders.merge(geography[["zip", "region"]],on="zip",how="left")
order_items["revenue"] = (order_items["quantity"] * order_items["unit_price"]- order_items["discount_amount"]) # Tính doanh thu trên từng item đặt
full_data = order_items.merge(orders_geo[["order_id", "region"]],on="order_id",how="left")
region_revenue = full_data.groupby("region", as_index=False)["revenue"].sum().sort_values("revenue", ascending=False)
print(region_revenue)
top_region = region_revenue.iloc[0]
print("\nRegion cao nhất:")
print(f"{top_region['region']} - {top_region['revenue']:.2f}")


    region       revenue
1     East  7.291151e+09
0  Central  4.719491e+09
2     West  3.670227e+09

Region cao nhất:
East - 7291150819.12


In [ ]:
# 8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?
import pandas as pd
ord = pd.read_csv("dataset/orders.csv")
ord = ord[ord["order_status"] == "cancelled"]["payment_method"].value_counts()
ord


payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

In [61]:
# 9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, 
# được định nghĩa là số bản ghi trong returns / số dòng trong order_items (join với products theo product_id)?
import pandas as pd
ord_i = pd.read_csv("dataset/order_items.csv", dtype={"promo_id_2": "string"} ); ret = pd.read_csv("dataset/returns.csv"); pro = pd.read_csv("dataset/products.csv")
ret = ret[["return_id","order_id","product_id"]]; pro = pro[["product_id", "size"]]; ord_i = ord_i[["order_id","product_id","quantity"]]
ret_pro = pd.merge(ret, pro, on = "product_id", how = "left")
return_counts = ret_pro.groupby("size").size()
ord_i_pro = pd.merge(ord_i, pro, on = "product_id", how = "left")
order_item_counts = ord_i_pro.groupby("size").size()
return_rate = (return_counts / order_item_counts * 100).sort_values(ascending=False)
return_rate

size
S     5.651527
L     5.624978
M     5.566010
XL    5.520010
dtype: float64

In [68]:
# 10. Trong payments.csv, 
# kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?
import pandas as pd
pay = pd.read_csv("dataset/payments.csv"); ord = pd.read_csv("dataset/orders.csv")
pay_ord = pd.merge(pay, ord, on = "order_id", how = "left")[["order_id", "payment_value","installments"]]
pay_value_mean = pay_ord.groupby("installments")["payment_value"].mean().sort_values(ascending=False)
pay_value_mean

installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64